In [ ]:
# !pip install -q "pathway" "sentence-transformers" "nltk"

In [ ]:
import os
import re
from typing import List, Dict, Any
from dataclasses import dataclass, asdict
import numpy as np
import nltk
from tqdm.auto import tqdm

# Ensure NLTK data is available
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

import pathway as pw
from sentence_transformers import SentenceTransformer

# --- CONFIG ---
# Use 'cuda' if available for 10x faster embedding
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 512 if DEVICE == "cuda" else 4
print(f"Running on: {DEVICE.upper()}")

@dataclass
class Chunk:
    chunk_id: int
    text: str
    start_pos: int
    end_pos: int
    book_name: str 

class NovelChunker:
    def __init__(self, chunk_size=450, overlap=60):
        self.chunk_size = chunk_size
        self.overlap = overlap

    def chunk(self, text: str, book_name: str) -> List[Chunk]:
        tokens = text.split()
        chunks = []
        start = 0
        cid = 0
        total_tokens = len(tokens)

        with tqdm(
            total=total_tokens,
            desc=f"Chunking [{book_name}]",
            unit="tok"
        ) as pbar:

            while start < total_tokens:
                end = min(start + self.chunk_size, total_tokens)
                chunk_tokens = tokens[start:end]
                text_str = " ".join(chunk_tokens)

                chunks.append(
                    Chunk(cid, text_str, start, end, book_name)
                )

                # ✅ Correct tqdm update (NO double counting)
                if cid == 0:
                    increment = end - start
                else:
                    increment = self.chunk_size - self.overlap

                pbar.update(min(increment, total_tokens - pbar.n))

                cid += 1
                start = end - self.overlap

        return chunks



class PathwayIndex:
    def __init__(self, embed_model="all-MiniLM-L6-v2"):
        # Load model to GPU if available
        self.embedder = SentenceTransformer(
            embed_model,
            device=DEVICE
        )
        self.embedder.eval()
        self.chunks_storage = [] 
        self.embeddings_storage = None

    def build(self, chunks: List[Chunk]):
        texts = [c.text for c in chunks]

        print(f"Embedding {len(texts)} chunks on {DEVICE}")
        print(f"Batch size = {BATCH_SIZE}")

        self.embeddings_storage = self.embedder.encode(
            texts,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        self.chunks_storage = chunks


    def search(self, query: str, top_k: int) -> List[Chunk]:
        if not self.chunks_storage: return []
            
        q_emb = self.embedder.encode(query, convert_to_numpy=True)
        
        # Fast Cosine Similarity using Numpy
        norm_q = np.linalg.norm(q_emb)
        norm_db = np.linalg.norm(self.embeddings_storage, axis=1)
        scores = np.dot(self.embeddings_storage, q_emb) / (norm_db * norm_q + 1e-10)
        
        top_indices = np.argsort(scores)[-top_k:][::-1]
        return [self.chunks_storage[i] for i in top_indices]

class NovelIndexer:
    def __init__(self):
        self.chunker = NovelChunker()
        self.indices: Dict[str, PathwayIndex] = {}

    def ingest(self, book_name: str, novel_path: str):
        print(f"Processing: {book_name}...")
        with open(novel_path, "r", encoding="utf-8") as f:
            text = f.read()

        chunks = self.chunker.chunk(text, book_name)
        index = PathwayIndex()
        index.build(chunks)
        self.indices[book_name] = index
        print(f"{book_name} Ready.")

    def retrieve_chunks_for_character(self, book_name: str, char_name: str, query: str, top_k: int) -> List[Chunk]:
        if book_name not in self.indices: return []
        
        # Retrieve wider pool
        all_chunks = self.indices[book_name].search(query, top_k * 4)
        
        # Filter
        name_pattern = re.compile(rf"\b{re.escape(char_name)}\b", re.IGNORECASE)
        filtered = [c for c in all_chunks if name_pattern.search(c.text)]
        
        return filtered[:top_k] if filtered else all_chunks[:top_k]

/home/Abo/miniconda3/envs/pathway/lib/python3.12/site-packages/fs/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore


Running on: CUDA


In [ ]:
import torch
torch.set_grad_enabled(False)

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
CUDA version: 11.8
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
TEST_DIR = "./data/Books/"
    
# 1. Check if directory exists
if not os.path.exists(TEST_DIR):
    print(f"Creating {TEST_DIR} and dummy files for testing...")
    os.makedirs(TEST_DIR, exist_ok=True)
    # Create dummy files if they don't exist
    with open(os.path.join(TEST_DIR, "In search of the castaways.txt"), "w") as f:
        f.write("Chapter 1. Lord Glenarvan finds a shark. Inside the shark is a bottle. " * 200)
    with open(os.path.join(TEST_DIR, "The Count of Monte Cristo.txt"), "w") as f:
        f.write("Edmond Dantes returned to Marseilles. He was happy to see Mercedes. " * 200)

# 2. Initialize System
indexer = NovelIndexer()
    
# 3. Ingest Books dynamically from the folder
files = [f for f in os.listdir(TEST_DIR) if f.endswith(".txt")]
    
for filename in files:
    book_name = filename.replace(".txt", "") # Clean name
    full_path = os.path.join(TEST_DIR, filename)
    indexer.ingest(book_name, full_path)
    
print("\n" + "="*40)
print("       SYSTEM READY FOR RETRIEVAL")
print("="*40 + "\n")

# 4. Test Case 1: The Count of Monte Cristo
target_book = "The Count of Monte Cristo"
target_char = "Dantes"
query = "Why was he returning to Marseilles?"
    
print(f"Searching in '{target_book}'...")
print(f"uery: {query}")
print(f"Character Focus: {target_char}")
    
results = indexer.retrieve_chunks_for_character(
    book_name=target_book,
    char_name=target_char,
    query=query,
    top_k=3
)
    
for i, r in enumerate(results):
    print(f"\n[Result {i+1}] (ID: {r.chunk_id})")
    print(f"Text: {r.text[:100]}...")

# 5. Test Case 2: In Search of the Castaways
target_book = "In search of the castaways"
target_char = "Glenarvan"
query = "What did they find inside the shark?"
    
print(f"\nSearching in '{target_book}'...")
print(f"Query: {query}")
    
results = indexer.retrieve_chunks_for_character(
    book_name=target_book,
    char_name=target_char,
    query=query,
    top_k=2
)
    
for i, r in enumerate(results):
    print(f"\n[Result {i+1}] (ID: {r.chunk_id})")
    print(f"Text: {r.text[:100]}...")

Processing: In search of the castaways...


Chunking [In search of the castaways]:   0%|          | 0/138830 [00:00<?, ?tok/s]